# 🧪 Sesión 14 · Hashing Seguro de Contraseñas (Bcrypt / Argon2)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué nunca se guarda una contraseña tal cual?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/registro  {"usuario":"ana","password":"secreta"}
HTTP/1.1 201 Created
{"usuario":"ana"}

POST /api/login  {"usuario":"ana","password":"secreta"}
HTTP/1.1 200 OK
{"usuario":"ana","autenticado":true}

POST /api/login  {"usuario":"ana","password":"incorrecta"}
HTTP/1.1 401 Unauthorized

La respuesta NUNCA contiene "password" ni el hash.
```

_El hash se guarda en el servidor; la contraseña original no se almacena en ningún lado._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Guardar contraseñas con bcrypt y salt.
- Verificar una contraseña sin descifrarla.
- Comprobar que la API nunca devuelve el hash ni la contraseña.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El registro responde sin filtrar la contraseña → HTTP 201
- [ ] El login correcto autentica → HTTP 200
- [ ] La contraseña incorrecta se rechaza → HTTP 401
- [ ] Se usa un algoritmo de hashing para contraseñas

## 🧩 El reto

Implementa registro y login con hashing seguro. La contraseña nunca debe viajar de vuelta al cliente.

**Requisitos del contrato**

- POST /api/registro recibe usuario y password, guarda solo el hash y responde 201.
- La respuesta de registro no incluye password ni hash.
- POST /api/login compara la contraseña con el hash y responde 200 o 401.
- Debes usar bcrypt (o passlib) para el hash.
- Un usuario repetido no debe sobrescribir la contraseña existente.

> **Pista conceptual:** No compares hashes con `==` si puedes usar la función de verificación del algoritmo. Bcrypt ya incluye el salt dentro del hash.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion14.md` explicando por qué no se puede «deshashear».


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [4]:
%%writefile app.py
import bcrypt
from fastapi import FastAPI, Request
from fastapi.encoders import jsonable_encoder
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from pydantic import BaseModel, ConfigDict, Field
from starlette.exceptions import HTTPException as StarletteHTTPException

app = FastAPI(title="Tu registro y login se verán así")


class Credenciales(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    usuario: str = Field(min_length=3, max_length=30)
    # bcrypt solo procesa los primeros 72 bytes, por eso el límite
    password: str = Field(min_length=6, max_length=64)


class UsuarioPublico(BaseModel):
    usuario: str


class SesionRespuesta(BaseModel):
    usuario: str
    autenticado: bool


# usuario -> hash bcrypt (nunca se guarda la contraseña en claro)
usuarios: dict[str, bytes] = {}

# Hash de relleno: permite comparar aunque el usuario no exista,
# para que el tiempo de respuesta no revele qué usuarios están registrados
_HASH_FALSO = bcrypt.hashpw(b"relleno", bcrypt.gensalt())


def generar_hash(password: str) -> bytes:
    return bcrypt.hashpw(password.encode("utf-8"), bcrypt.gensalt())


def verificar_password(password: str, hash_guardado: bytes) -> bool:
    return bcrypt.checkpw(password.encode("utf-8"), hash_guardado)


@app.post("/api/registro", status_code=201, response_model=UsuarioPublico)
def registro(datos: Credenciales):
    if datos.usuario in usuarios:
        raise StarletteHTTPException(status_code=409, detail="El usuario ya existe")
    usuarios[datos.usuario] = generar_hash(datos.password)
    return {"usuario": datos.usuario}


@app.post("/api/login", response_model=SesionRespuesta)
def login(datos: Credenciales):
    hash_guardado = usuarios.get(datos.usuario)
    valido = verificar_password(datos.password, hash_guardado or _HASH_FALSO)
    if hash_guardado is None or not valido:
        raise StarletteHTTPException(status_code=401, detail="Credenciales inválidas")
    return {"usuario": datos.usuario, "autenticado": True}


def sobre_error(estado: int, codigo: str, mensaje: str, detalles=None) -> JSONResponse:
    cuerpo = {"error": {"codigo": codigo, "mensaje": mensaje}}
    if detalles is not None:
        cuerpo["error"]["detalles"] = detalles
    return JSONResponse(status_code=estado, content=cuerpo)


@app.exception_handler(StarletteHTTPException)
async def manejar_http(request: Request, exc: StarletteHTTPException):
    codigos = {
        401: "NO_AUTENTICADO",
        404: "NO_ENCONTRADO",
        405: "METODO_NO_PERMITIDO",
        409: "CONFLICTO",
    }
    return sobre_error(exc.status_code, codigos.get(exc.status_code, "ERROR_HTTP"), str(exc.detail))


@app.exception_handler(RequestValidationError)
async def manejar_validacion(request: Request, exc: RequestValidationError):
    # Solo se devuelven campo y mensaje; nunca el valor enviado (podría ser la contraseña)
    detalles = jsonable_encoder(
        [
            {"campo": ".".join(str(p) for p in e["loc"][1:]), "mensaje": e["msg"]}
            for e in exc.errors()
        ]
    )
    return sobre_error(422, "VALIDACION", "Los datos enviados no son válidos", detalles)


Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [5]:
from verificar import probar_http

casos = [{'descripcion': 'El registro responde sin filtrar la contraseña',
  'metodo': 'POST',
  'ruta': '/api/registro',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 201,
  'json_contiene': {'usuario': 'ana'},
  'no_contiene': ['password', 'hash']},
 {'descripcion': 'El login correcto autentica',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana', 'autenticado': True}},
 {'descripcion': 'La contraseña incorrecta se rechaza',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'incorrecta'},
  'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 3 casos · app.py
✅ 01. El registro responde sin filtrar la contraseña
✅ 02. El login correcto autentica
✅ 03. La contraseña incorrecta se rechaza
------------------------------------------------------------------------------
RESULTADO: 3/3 casos correctos. ¡Resultado logrado!


In [6]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["bcrypt|passlib"], descripcion="Se usa un algoritmo de hashing para contraseñas"))
resumen(resultados)


✅ Se usa un algoritmo de hashing para contraseñas
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion14.md` a `auditorias/auditoria_sesion14.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
